In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
Yield=pd.read_csv('crop_yield.csv')
Yield.head()

,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,Arecanut,1997,Whole Year,Assam,73814.0,56708,2051.4,7024878.38,22882.34,0.796087
1,Arhar/Tur,1997,Kharif,Assam,6637.0,4685,2051.4,631643.29,2057.47,0.710435
2,Castor seed,1997,Kharif,Assam,796.0,22,2051.4,75755.32,246.76,0.238333
3,Coconut,1997,Whole Year,Assam,19656.0,126905000,2051.4,1870661.52,6093.36,5238.051739
4,Cotton(lint),1997,Kharif,Assam,1739.0,794,2051.4,165500.63,539.09,0.420909


In [4]:
Yield.shape

(19689, 10)

In [6]:
drop_cols = ["Crop_Year", "Season", "State", "Pesticide"]
Yield= Yield.drop(columns=[c for c in drop_cols if c in Yield.columns])

In [7]:
Yield.head()

,Crop,Area,Production,Annual_Rainfall,Fertilizer,Yield
0,Arecanut,73814.0,56708,2051.4,7024878.38,0.796087
1,Arhar/Tur,6637.0,4685,2051.4,631643.29,0.710435
2,Castor seed,796.0,22,2051.4,75755.32,0.238333
3,Coconut,19656.0,126905000,2051.4,1870661.52,5238.051739
4,Cotton(lint),1739.0,794,2051.4,165500.63,0.420909


In [8]:
Yield.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19689 entries, 0 to 19688
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Crop             19689 non-null  object 
 1   Area             19689 non-null  float64
 2   Production       19689 non-null  int64  
 3   Annual_Rainfall  19689 non-null  float64
 4   Fertilizer       19689 non-null  float64
 5   Yield            19689 non-null  float64
dtypes: float64(4), int64(1), object(1)
memory usage: 923.1+ KB


In [9]:
Yield.isnull().sum()

Crop               0
Area               0
Production         0
Annual_Rainfall    0
Fertilizer         0
Yield              0
dtype: int64

In [10]:
Yield.describe()

,Area,Production,Annual_Rainfall,Fertilizer,Yield
count,1.968900e+04,1.968900e+04,19689.000000,1.968900e+04,19689.000000
mean,1.799266e+05,1.643594e+07,1437.755177,2.410331e+07,79.954009
std,7.328287e+05,2.630568e+08,816.909589,9.494600e+07,878.306193
min,5.000000e-01,0.000000e+00,301.300000,5.417000e+01,0.000000
25%,1.390000e+03,1.393000e+03,940.700000,1.880146e+05,0.600000
50%,9.317000e+03,1.380400e+04,1247.600000,1.234957e+06,1.030000
75%,7.511200e+04,1.227180e+05,1643.700000,1.000385e+07,2.388889
max,5.080810e+07,6.326000e+09,6552.700000,4.835407e+09,21105.000000


In [14]:
HA_TO_ACRE = 2.47

Yield["Area_acres"] = Yield["Area"] * HA_TO_ACRE
Yield["Fertilizer_per_acre"] = Yield["Fertilizer"] / Yield["Area_acres"]
Yield["Rainfall_mm"] = Yield["Annual_Rainfall"]
Yield["Yield_quintal_per_acre"] = Yield["Yield"] * (10 / HA_TO_ACRE)

In [15]:
for col in ["Fertilizer_per_acre", "Yield_quintal_per_acre"]:
    low, high = Yield[col].quantile([0.01, 0.99])
    Yield= Yield[(Yield[col] >= low) & (Yield[col] <= high)]

In [16]:
np.random.seed(42)
Yield["Soil_Quality"] = np.random.choice(
    ["Good", "Medium", "Poor"], size=len(Yield), p=[0.4, 0.4, 0.2]
)
soil_factor = Yield["Soil_Quality"].map({"Good": 1.10, "Medium": 1.00, "Poor": 0.90})
Yield["Yield_quintal_per_acre"] = Yield["Yield_quintal_per_acre"] * soil_factor

In [17]:
new_yield = Yield[[
    "Crop",
    "Area_acres",
    "Fertilizer_per_acre",
    "Rainfall_mm",
    "Soil_Quality",
    "Yield_quintal_per_acre",
]].rename(columns={
    "Area_acres": "Area",
    "Fertilizer_per_acre": "Fertilizer",
    "Rainfall_mm": "Rainfall",
    "Yield_quintal_per_acre": "Yield",
})

In [18]:
print("\nFinal training columns:", list(new_yield.columns))
print(new_yield.head())


Final training columns: ['Crop', 'Area', 'Fertilizer', 'Rainfall', 'Soil_Quality', 'Yield']
           Crop       Area  Fertilizer  Rainfall Soil_Quality     Yield
0      Arecanut  182320.58   38.530364    2051.4         Good  3.545327
1     Arhar/Tur   16393.39   38.530364    2051.4         Poor  2.588629
2   Castor seed    1966.12   38.530364    2051.4       Medium  0.964912
4  Cotton(lint)    4295.33   38.530364    2051.4       Medium  1.704085
5  Dry chillies   33559.89   38.530364    2051.4         Good  2.866397


In [19]:
crop_encoder = LabelEncoder()
soil_encoder = LabelEncoder()

new_yield["Crop_enc"] = crop_encoder.fit_transform(new_yield["Crop"])
new_yield["Soil_enc"] = soil_encoder.fit_transform(new_yield["Soil_Quality"])

x= new_yield[["Crop_enc", "Area", "Fertilizer", "Rainfall", "Soil_enc"]]
y = new_yield["Yield"]

In [20]:
x_train, x_test, y_train, y_test = train_test_split( x, y, test_size=0.2, random_state=42)

In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor

models = {
    'Linear Regression': LinearRegression(),
    'KNN': KNeighborsRegressor(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Extra Trees': ExtraTreesRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42)
}

for name, model in models.items():
    model.fit(x_train, y_train)
    ypred = model.predict(x_test)

    r2 = r2_score(y_test, ypred)
    mae = mean_absolute_error(y_test, ypred)

    print(f"{name}")
    print(f"R² Score: {r2:.4f}")
    print(f"MAE: {mae:.2f}")
    print("-" * 40)

Linear Regression
R² Score: 0.0195
MAE: 20.89
----------------------------------------
KNN
R² Score: -0.0925
MAE: 21.81
----------------------------------------
Decision Tree
R² Score: 0.7954
MAE: 5.97
----------------------------------------
Random Forest
R² Score: 0.8640
MAE: 5.07
----------------------------------------
Extra Trees
R² Score: 0.8286
MAE: 5.77
----------------------------------------
XGBoost
R² Score: 0.8489
MAE: 6.42
----------------------------------------


In [23]:
final_model = RandomForestRegressor(random_state=42)
final_model.fit(x, y)

RandomForestRegressor(random_state=42)

In [37]:
def predict_yield(crop, area_acres, fertilizer_kg, rainfall_mm, soil_quality):

    # Encode categorical inputs
    crop_code = crop_encoder.transform([crop])[0]
    soil_code = soil_encoder.transform([soil_quality])[0]

    # Convert total fertilizer to fertilizer per acre
    fert_per_acre = fertilizer_kg / area_acres

    # Create input dataframe (same order as training)
    input_row = pd.DataFrame([{
        "Crop_enc": crop_code,
        "Area": area_acres,
        "Fertilizer": fert_per_acre,
        "Rainfall": rainfall_mm,
        "Soil_enc": soil_code
    }])

    # Predict yield per acre
    yield_per_acre = final_model.predict(input_row)[0]

    # Calculate total production
    total_yield = yield_per_acre * area_acres

    return round(yield_per_acre, 1), round(total_yield, 1)

In [38]:
result = predict_yield(
    crop="Rice",
    area_acres=10,
    fertilizer_kg=300,
    rainfall_mm=1200,
    soil_quality="Good"
)

print("Yield per acre:", result[0], "quintals")
print("Total yield:", result[1], "quintals")

Yield per acre: 7.4 quintals
Total yield: 73.8 quintals


In [40]:
import joblib
joblib.dump(final_model, "yield_model.pkl")
joblib.dump(crop_encoder, "crop_encoder.pkl")
joblib.dump(soil_encoder, "soil_encoder.pkl")
print("Model and encoders saved successfully!")

Model and encoders saved successfully!
